# Assignment 1 - Part 2 — Backward Pass — **SOLUTION**
**Introduction to Deep Learning, THWS**

> **Instructor use only — do not distribute to students.**

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from helpers import load_data, numerical_gradient, grad_checker

X, y = load_data()
print(f'Data loaded: X={X.shape}, y={y.shape}')

Data loaded: X=torch.Size([500, 4]), y=torch.Size([500, 1])


---
## Section 1 — MSE backward

Recall the MSE loss for a single prediction:
$$L = (\hat{y} - y)^2$$

MSE is the **root** of our computation graph — there is no upstream gradient.
This means the local and global gradients are always identical:
$$\frac{\partial L}{\partial \hat{y}} = 2(\hat{y} - y)$$
We therefore have a single backward function for MSE.

**Task 1.1** Implement `mse_backward_scalar` and `mse_backward` in `functions.py`.

In [2]:
from functions import mse_backward_scalar

x0     = X[0, 0].item()
y0     = y[0].item()
theta  = torch.tensor([0.0, 1.0])
y_pred = theta[1] * x0 + theta[0]

g = mse_backward_scalar(y_pred, y0)
print(f'y_pred: {y_pred:.4f},  y: {y0:.4f}')
print(f'dL/d(y_pred): {g:.4f}')

y_pred: 0.5304,  y: -0.9119
dL/d(y_pred): 2.8846


In [3]:
# verify with numerical gradient
f_scalar = lambda yp: torch.tensor((yp.item() - y0) ** 2)
g_num    = numerical_gradient(f_scalar, torch.tensor([y_pred]))
grad_checker(torch.tensor([g]), g_num, name='mse_backward_scalar')

[mse_backward_scalar] max absolute error: 2.89e-04
[mse_backward_scalar] Gradient check PASSED.


**Question 1.1** What does the sign of the gradient tell you about whether a prediction is too high or too low?

*Answer:* If $\hat{y} > y$, then $\hat{y} - y > 0$ and the gradient is positive — the loss increases as the prediction increases, so we should decrease it. If $\hat{y} < y$, the gradient is negative — we should increase the prediction. The gradient always points in the direction that increases the loss; subtracting it moves the prediction toward the target.

Now extend to the batch case. The batch MSE is:
$$L = \frac{1}{N} \sum_{i=1}^N (\hat{y}^{(i)} - y^{(i)})^2$$

**Task 1.2** Derive $\partial L / \partial \hat{y}^{(i)}$ for each example, then implement `mse_backward` in `functions.py`.

In [4]:
from functions import mse_backward
from layers import linear_forward

torch.manual_seed(0)
theta_1 = torch.randn(1, 4)
theta_0 = torch.zeros(1, 1)
y_pred  = linear_forward(X, theta_1, theta_0)

g = mse_backward(y_pred, y)
print(f'gradient shape:  {g.shape}')
print(f'first 5 values:  {g[:5].squeeze().tolist()}')

gradient shape:  torch.Size([500, 1])
first 5 values:  [0.0016534074675291777, 0.0017999461852014065, 0.005688684526830912, -0.0037710745818912983, -0.004854164551943541]


In [5]:
# verify with numerical gradient on a small batch
from layers import mse_forward

X_small, y_small = X[:5], y[:5]
torch.manual_seed(0)
yp_small = linear_forward(X_small, torch.randn(1, 4), torch.zeros(1, 1))
g_small  = mse_backward(yp_small, y_small)
g_num    = numerical_gradient(lambda yp: mse_forward(yp, y_small), yp_small)
grad_checker(g_small, g_num, name='mse_backward')

[mse_backward] max absolute error: 4.06e-04
[mse_backward] Gradient check PASSED.


---
## Section 2 — Linear backward

Consider the linear layer: $Z = X \Theta_1^\top + \Theta_0$

Unlike MSE, this node sits in the middle of the graph. We split the backward pass:
1. **Local gradients** (`_lgrad`) — partial derivatives of $Z$ w.r.t. its inputs
2. **Global gradients** (`_ggrad`) — apply the chain rule with upstream $G_{out}$. In the scalar case: elementwise `gout * lgrad`. In the matrix case: matrix products.

**Task 2.1** Derive the three local gradients for $z = \theta_1 x + \theta_0$ on paper.

**Question 2.1** What are the three local gradients?

*Answer:* $\partial z / \partial \theta_1 = x$, $\partial z / \partial \theta_0 = 1$, $\partial z / \partial x = \theta_1$.

**Task 2.2** Implement `linear_lgrad_scalar` and `linear_ggrad_scalar` in `functions.py`.

In [6]:
from functions import linear_lgrad_scalar, linear_ggrad_scalar

x     = X[0, 0]
theta = torch.tensor([0.0, 1.0])   # (theta_0, theta_1)
gout  = torch.tensor(1.0)          # upstream gradient — start with 1

# step 1: local gradients
lgrad_theta_0, lgrad_theta_1, lgrad_x = linear_lgrad_scalar(x, theta)
print('Local gradients:')
print(f'  dz/d(theta_0) = {lgrad_theta_0:.4f}')
print(f'  dz/d(theta_1) = {lgrad_theta_1:.4f}')
print(f'  dz/dx         = {lgrad_x:.4f}')

# step 2: global gradients — gout=1 so global = local
ggrad_theta_0, ggrad_theta_1, ggrad_x = linear_ggrad_scalar(gout, x, theta)
theta.g = torch.stack([ggrad_theta_0, ggrad_theta_1])
x.g     = ggrad_x
print('\nGlobal gradients (gout=1):')
print(f'  theta.g = {theta.g}')
print(f'  x.g     = {x.g:.4f}')

Local gradients:
  dz/d(theta_0) = 1.0000
  dz/d(theta_1) = 0.5304
  dz/dx         = 1.0000

Global gradients (gout=1):
  theta.g = tensor([1.0000, 0.5304])
  x.g     = 1.0000


In [7]:
# now use actual upstream gradient from mse_backward_scalar
y0     = y[0].item()
y_pred = (theta[1] * x + theta[0]).item()
gout   = torch.tensor(mse_backward_scalar(y_pred, y0))

ggrad_theta_0, ggrad_theta_1, ggrad_x = linear_ggrad_scalar(gout, x, theta)
theta.g = torch.stack([ggrad_theta_0, ggrad_theta_1])
x.g     = ggrad_x

print(f'gout: {gout.item():.4f}')
print(f'theta.g: {theta.g}')
print(f'x.g:     {x.g:.4f}')

f_loss = lambda t: torch.tensor((t[1].item() * x.item() + t[0].item() - y0) ** 2)
grad_checker(theta.g, numerical_gradient(f_loss, theta), name='linear scalar')

gout: 2.8846
theta.g: tensor([2.8846, 1.5300])
x.g:     2.8846
[linear scalar] max absolute error: 5.79e-04
[linear scalar] Gradient check PASSED.


**Task 2.3** Now extend to the batch case $Z = X \Theta_1^\top + \Theta_0$.

In the matrix case the chain rule with upstream $G_{out} \in \mathbb{R}^{N \times d_{out}}$ produces matrix products:
$$\frac{\partial L}{\partial \Theta_1} = G_{out}^\top X \qquad
\frac{\partial L}{\partial \Theta_0} = \sum_i G_{out}^{(i)} \qquad
\frac{\partial L}{\partial X} = G_{out} \, \Theta_1$$

**Question 2.2** What are the shapes of $G_{out}$, $\partial L / \partial \Theta_1$, $\partial L / \partial \Theta_0$, and $\partial L / \partial X$?

*Answer:* $G_{out} \in \mathbb{R}^{N \times d_{out}}$, $\partial L / \partial \Theta_1 \in \mathbb{R}^{d_{out} \times d_{in}}$, $\partial L / \partial \Theta_0 \in \mathbb{R}^{1 \times d_{out}}$, $\partial L / \partial X \in \mathbb{R}^{N \times d_{in}}$.

**Question 2.3** Why do we need $\partial L / \partial X$ in addition to the parameter gradients?

*Answer:* It is the gradient passed to the preceding layer — without it the backward pass cannot propagate beyond a single layer.

**Task 2.4** Implement `linear_lgrad` and `linear_ggrad` in `functions.py`.

In [8]:
from functions import linear_lgrad, linear_ggrad

torch.manual_seed(0)
theta_1 = torch.randn(1, 4)
theta_0 = torch.zeros(1, 1)
y_pred  = linear_forward(X, theta_1, theta_0)
gout    = mse_backward(y_pred, y)

# step 1: local gradients
lgrad_t1, lgrad_t0, lgrad_ins = linear_lgrad(X, theta_1, theta_0)
print('Local gradient factors:')
print(f'  lgrad_theta_1 factor: {lgrad_t1.shape}')
print(f'  lgrad_theta_0 factor: {lgrad_t0.shape}')
print(f'  lgrad_ins:            {lgrad_ins.shape}')

# step 2: global gradients via linear_ggrad
theta_1.g, theta_0.g, X.g = linear_ggrad(gout, X, theta_1, theta_0)
print('\nGlobal gradients:')
print(f'  theta_1.g: {theta_1.g.shape}')
print(f'  theta_0.g: {theta_0.g.shape}')
print(f'  X.g:       {X.g.shape}')

Local gradient factors:
  lgrad_theta_1 factor: torch.Size([500, 4])
  lgrad_theta_0 factor: torch.Size([500, 1])
  lgrad_ins:            torch.Size([1, 4])

Global gradients:
  theta_1.g: torch.Size([1, 4])
  theta_0.g: torch.Size([1, 1])
  X.g:       torch.Size([500, 4])


In [9]:
# verify
from layers import mse_forward

f_t1 = lambda t1: mse_forward(linear_forward(X, t1, theta_0), y)
f_t0 = lambda t0: mse_forward(linear_forward(X, theta_1, t0), y)
f_x  = lambda x:  mse_forward(linear_forward(x, theta_1, theta_0), y)

grad_checker(theta_1.g, numerical_gradient(f_t1, theta_1), name='theta_1')
grad_checker(theta_0.g, numerical_gradient(f_t0, theta_0), name='theta_0')
grad_checker(X.g,       numerical_gradient(f_x, X),        name='X')

[theta_1] max absolute error: 5.61e-03
[theta_1] Gradient check PASSED.
[theta_0] max absolute error: 1.27e-07
[theta_0] Gradient check PASSED.
[X] max absolute error: 5.54e-03
[X] Gradient check PASSED.


---
## Section 3 — ReLU backward

Recall the ReLU: $a = \text{relu}(z) = \max(0, z)$

**Task 3.1** Derive the local gradient $\partial a / \partial z$ for $z > 0$, $z < 0$, and $z = 0$.

**Question 3.1** What is the local gradient for each case?

*Answer:* $da/dz = 1$ if $z > 0$, and $0$ if $z \leq 0$.

**Task 3.2** Implement `relu_lgrad_scalar`, `relu_ggrad_scalar`, `relu_lgrad`, and `relu_ggrad` in `functions.py`.

In [10]:
from functions import relu_lgrad_scalar, relu_ggrad_scalar

for val in [2.0, -1.5, 0.0]:
    z    = torch.tensor(val)
    gout = torch.tensor(1.0)

    # step 1: local gradient
    lgrad = relu_lgrad_scalar(z)

    # step 2: global gradient
    z.g = relu_ggrad_scalar(gout, z)

    print(f'z={val:5.1f}  lgrad={lgrad:.1f}  z.g={z.g:.1f}')

z=  2.0  lgrad=1.0  z.g=1.0
z= -1.5  lgrad=0.0  z.g=0.0
z=  0.0  lgrad=0.0  z.g=0.0


Expected output:
```
z=  2.0  lgrad=1.0  z.g=1.0
z= -1.5  lgrad=0.0  z.g=0.0
z=  0.0  lgrad=0.0  z.g=0.0
```

**Question 3.2** What happens to the gradient flowing through an inactive unit (pre-activation $< 0$)? What does this mean for the parameters upstream of that unit?

*Answer:* The gradient is multiplied by zero — it is completely blocked. Parameters in earlier layers that only affect the network through that inactive unit receive zero gradient and are not updated. This is known as the *dying ReLU* problem: if a unit is always inactive across all training examples, it can never be recovered by gradient descent.

---
## Section 4 — Backward pass through the classes

The `Linear` and `ReLU` classes in `layers_backward.py` implement `backward` using
`linear_ggrad` and `relu_ggrad`. Each method assigns the returned values to `.g` attributes
and returns `self.ins.g` to pass the gradient to the preceding layer.

**Task 4.1** Implement `Linear.backward` in `layers_backward.py`.

In [11]:
from layers_backward import Linear, ReLU, Model

torch.manual_seed(0)
layer = Linear(theta_1=torch.randn(1, 4), theta_0=torch.zeros(1, 1))
y_pred = layer.forward(X)
gout   = mse_backward(y_pred, y)

layer.backward(gout)

print(f'layer.theta_1.g shape: {layer.theta_1.g.shape}')   # (1, 4)
print(f'layer.theta_0.g shape: {layer.theta_0.g.shape}')   # (1, 1)
print(f'layer.ins.g shape:     {layer.ins.g.shape}')        # (500, 4)

layer.theta_1.g shape: torch.Size([1, 4])
layer.theta_0.g shape: torch.Size([1, 1])
layer.ins.g shape:     torch.Size([500, 4])


In [12]:
# verify against numerical gradient
torch.manual_seed(0)
layer = Linear(theta_1=torch.randn(1, 4), theta_0=torch.zeros(1, 1))
layer.forward(X)
layer.backward(mse_backward(layer.outs, y))

f_t1 = lambda t1: mse_forward(linear_forward(X, t1, layer.theta_0), y)
f_t0 = lambda t0: mse_forward(linear_forward(X, layer.theta_1, t0), y)

grad_checker(layer.theta_1.g, numerical_gradient(f_t1, layer.theta_1), name='Linear.theta_1')
grad_checker(layer.theta_0.g, numerical_gradient(f_t0, layer.theta_0), name='Linear.theta_0')

[Linear.theta_1] max absolute error: 5.61e-03
[Linear.theta_1] Gradient check PASSED.
[Linear.theta_0] max absolute error: 1.27e-07
[Linear.theta_0] Gradient check PASSED.


**Task 4.2** Implement `ReLU.backward` in `layers_backward.py`.

In [13]:
relu = ReLU()
z    = torch.tensor([[-1.5,  0.5,  2.0],
                     [ 1.0, -0.3,  0.0]])
a    = relu.forward(z)
gout = torch.ones_like(a)   # upstream gradient all ones

relu.backward(gout)
print(f'z:\n{z}')
print(f'z.g:\n{relu.ins.g}')

z:
tensor([[-1.5000,  0.5000,  2.0000],
        [ 1.0000, -0.3000,  0.0000]])
z.g:
tensor([[0., 1., 1.],
        [1., 0., 0.]])


**Task 4.3** Implement `Model.backward` in `layers_backward.py`.

**Question 4.1** Why must the backward pass run through layers in reverse order?

*Answer:* The chain rule requires multiplying gradients in reverse order of the forward pass. To compute the gradient at layer $l$, we need the upstream gradient from layer $l+1$, which in turn needs the gradient from layer $l+2$, and so on. Each layer's backward pass depends on the result of the layer ahead of it in the backward direction — i.e., the layer after it in the forward direction.

In [14]:
torch.manual_seed(0)
model = Model([
    Linear(theta_1=torch.randn(8, 4), theta_0=torch.zeros(1, 8)),
    ReLU(),
    Linear(theta_1=torch.randn(1, 8), theta_0=torch.zeros(1, 1)),
])

y_pred = model.forward(X)
gout   = mse_backward(y_pred, y)
model.backward(gout)

# check gradients exist for all linear layers
for i, layer in enumerate(model.layers):
    if isinstance(layer, Linear):
        print(f'Layer {i}: theta_1.g shape={layer.theta_1.g.shape}, theta_0.g shape={layer.theta_0.g.shape}')

Layer 0: theta_1.g shape=torch.Size([8, 4]), theta_0.g shape=torch.Size([1, 8])
Layer 2: theta_1.g shape=torch.Size([1, 8]), theta_0.g shape=torch.Size([1, 1])


In [15]:
# verify all parameter gradients against PyTorch autograd
torch.manual_seed(0)
t1_0 = torch.randn(8, 4, requires_grad=True)
t0_0 = torch.zeros(1, 8, requires_grad=True)
t1_1 = torch.randn(1, 8, requires_grad=True)
t0_1 = torch.zeros(1, 1, requires_grad=True)

# autograd forward
z0   = X @ t1_0.T + t0_0
a0   = z0.clamp(0)
z1   = a0 @ t1_1.T + t0_1
loss = torch.mean((z1 - y) ** 2)
loss.backward()

# compare
l0, l1 = model.layers[0], model.layers[2]
grad_checker(l0.theta_1.g, t1_0.grad, name='layer0 theta_1')
grad_checker(l0.theta_0.g, t0_0.grad, name='layer0 theta_0')
grad_checker(l1.theta_1.g, t1_1.grad, name='layer1 theta_1')
grad_checker(l1.theta_0.g, t0_1.grad, name='layer1 theta_0')

[layer0 theta_1] max absolute error: 0.00e+00
[layer0 theta_1] Gradient check PASSED.
[layer0 theta_0] max absolute error: 0.00e+00
[layer0 theta_0] Gradient check PASSED.
[layer1 theta_1] max absolute error: 0.00e+00
[layer1 theta_1] Gradient check PASSED.
[layer1 theta_0] max absolute error: 0.00e+00
[layer1 theta_0] Gradient check PASSED.


---
## Section 5 — Deeper network

Build the following architecture and verify the full forward and backward pass:

$$\text{Linear}(4 \to 5) \to \text{ReLU} \to \text{Linear}(5 \to 10) \to \text{ReLU} \to \text{Linear}(10 \to 4) \to \text{ReLU} \to \text{Linear}(4 \to 1)$$

**Task 5.1** Define `deep_model` with the architecture above.

In [16]:
torch.manual_seed(0)

deep_model = Model([
    Linear(theta_1=torch.randn(5,  4) * 0.1, theta_0=torch.zeros(1,  5)),
    ReLU(),
    Linear(theta_1=torch.randn(10, 5) * 0.1, theta_0=torch.zeros(1, 10)),
    ReLU(),
    Linear(theta_1=torch.randn(4, 10) * 0.1, theta_0=torch.zeros(1,  4)),
    ReLU(),
    Linear(theta_1=torch.randn(1,  4) * 0.1, theta_0=torch.zeros(1,  1)),
])

# forward + backward
y_pred = deep_model.forward(X)
print(f'Output shape: {y_pred.shape}')   # expect (500, 1)
deep_model.backward(mse_backward(y_pred, y))
print('Backward pass completed.')

for i, layer in enumerate(deep_model.layers):
    if isinstance(layer, Linear):
        print(f'Layer {i}: theta_1.g={layer.theta_1.g.shape}')

Output shape: torch.Size([500, 1])
Backward pass completed.
Layer 0: theta_1.g=torch.Size([5, 4])
Layer 2: theta_1.g=torch.Size([10, 5])
Layer 4: theta_1.g=torch.Size([4, 10])
Layer 6: theta_1.g=torch.Size([1, 4])


In [17]:
# verify against PyTorch autograd — rebuild same network with requires_grad
from layers import mse_forward

torch.manual_seed(0)
t1_0 = torch.randn(5,  4, requires_grad=True) * 0.1
t0_0 = torch.zeros(1,  5, requires_grad=True)
t1_1 = torch.randn(10, 5, requires_grad=True) * 0.1
t0_1 = torch.zeros(1, 10, requires_grad=True)
t1_2 = torch.randn(4, 10, requires_grad=True) * 0.1
t0_2 = torch.zeros(1,  4, requires_grad=True)
t1_3 = torch.randn(1,  4, requires_grad=True) * 0.1
t0_3 = torch.zeros(1,  1, requires_grad=True)

# autograd forward — no ReLU after last linear layer
a0 = (X @ t1_0.T + t0_0).clamp(0)
a1 = (a0 @ t1_1.T + t0_1).clamp(0)
a2 = (a1 @ t1_2.T + t0_2).clamp(0)
yp =  a2 @ t1_3.T + t0_3
torch.mean((yp - y) ** 2).backward()

# compare — autograd returns None for zero gradients, treat as zeros
def to_tensor(g, shape):
    return g if g is not None else torch.zeros(shape)

linear_layers = [l for l in deep_model.layers if isinstance(l, Linear)]
for j, (layer, t1_ref, t0_ref) in enumerate(
    zip(linear_layers, [t1_0, t1_1, t1_2, t1_3], [t0_0, t0_1, t0_2, t0_3])
):
    grad_checker(layer.theta_1.g, to_tensor(t1_ref.grad, layer.theta_1.shape), name=f'layer {j} theta_1')
    grad_checker(layer.theta_0.g, to_tensor(t0_ref.grad, layer.theta_0.shape), name=f'layer {j} theta_0')

[layer 0 theta_1] max absolute error: 9.00e-04
[layer 0 theta_1] Gradient check PASSED.
[layer 0 theta_0] max absolute error: 0.00e+00
[layer 0 theta_0] Gradient check PASSED.
[layer 1 theta_1] max absolute error: 9.29e-04
[layer 1 theta_1] Gradient check PASSED.
[layer 1 theta_0] max absolute error: 2.89e-04
[layer 1 theta_0] Gradient check PASSED.
[layer 2 theta_1] max absolute error: 1.57e-03
[layer 2 theta_1] Gradient check PASSED.
[layer 2 theta_0] max absolute error: 1.44e-03
[layer 2 theta_0] Gradient check PASSED.
[layer 3 theta_1] max absolute error: 1.96e-03
[layer 3 theta_1] Gradient check PASSED.
[layer 3 theta_0] max absolute error: 0.00e+00
[layer 3 theta_0] Gradient check PASSED.


/tmp/ipykernel_93476/46044926.py:29: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  grad_checker(layer.theta_1.g, to_tensor(t1_ref.grad, layer.theta_1.shape), name=f'layer {j} theta_1')


**Question 5.1** Does the number of layers affect whether the gradient check passes?

*Answer:* No — as long as each layer's local gradients are correct and the chain rule is applied in the right order, the gradient check passes regardless of depth. Errors would accumulate only if individual layer gradients were wrong.